# #7DaysOfCode — Agentes de IA & Engenharia de Prompts
## Resolução Técnica — Dia 5: Memória Conversacional Multi-Sessão e Janela de Contexto

Este notebook apresenta a implementação técnica do **Dia 5**, adicionando gerenciamento de memória por sessão (`session_id` com `InMemoryChatMessageHistory`) e controle de janela de contexto (`trim_messages`) ao Agente ReAct da **NovaMart**.

### Escopo Técnico desta Implementação
- Isolamento de histórico conversacional por `session_id` (`ARMAZEM_SESSOES_CHAT` e `obter_historico_sessao`);
- Estratégia de janela deslizante (`recortar_janela_historico`) preservando a `SystemMessage` e a integridade dos pares `AIMessage(tool_calls)` $\rightarrow$ `ToolMessage`;
- Integração de histórico multi-turno ao loop ReAct (`conversar_com_clara_dia5`).

---

## Etapa 1: Preparação do Ambiente e Conexão Segura

Instalamos as bibliotecas da stack (`langchain`, `langchain-core`, `langchain-google-genai`, `pydantic` e `python-dotenv`) e configuramos a `GOOGLE_API_KEY`.

In [ ]:
# Nota: No Google Colab, pode aparecer um aviso sobre versão do 'google-auth', mas a instalação funciona normalmente!
!pip install -q -U langchain langchain-core langchain-google-genai pydantic python-dotenv

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
        print("Chave GOOGLE_API_KEY carregada com sucesso dos Secrets do Google Colab!")
    except Exception:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Digite sua GOOGLE_API_KEY (Google AI Studio): ")
        print("Chave configurada com sucesso para esta sessão!")
else:
    print("GOOGLE_API_KEY já está configurada no ambiente!")

---
## Etapa 2: Base Herdada (Dias 1 a 4) — Modelo, Dados (`pedidos_novamart.json` + Estoque) e Ferramentas

Na célula abaixo, carregamos a base construída nos dias anteriores:
- O modelo **`gemini-2.5-flash`** (`temperature=0.0`);
- O **`BANCO_PEDIDOS_NOVAMART`** lido do arquivo externo [`../data/pedidos_novamart.json`](../data/pedidos_novamart.json) e a tabela **`ESTOQUE_CD_NOVAMART`**;
- As 3 ferramentas operacionais (`consultar_pedido_novamart`, `verificar_estoque_produto_cd` e `abrir_protocolo_logistica_erp`).

In [ ]:
import json
from pathlib import Path
import unicodedata
from typing import Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

# Inicializando o modelo Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.0
)

# =====================================================================
# HERANÇA DOS DIAS 3 E 4: Banco de Pedidos e Estoque do CD da NovaMart
# =====================================================================
def localizar_arquivo_dados(nome_arquivo: str) -> Path:
    """Localiza o arquivo na pasta '../data/', 'data/' ou no diretório atual do Colab."""
    for candidato in [Path(f"../data/{nome_arquivo}"), Path(f"data/{nome_arquivo}"), Path(nome_arquivo)]:
        if candidato.exists():
            return candidato
    raise FileNotFoundError(
        f"Arquivo '{nome_arquivo}' não encontrado! Certifique-se de que ele está em '../data/{nome_arquivo}' "
        f"ou faça o upload de '{nome_arquivo}' na aba de Arquivos do Google Colab."
    )

caminho_pedidos = localizar_arquivo_dados("pedidos_novamart.json")
BANCO_PEDIDOS_NOVAMART = json.loads(caminho_pedidos.read_text(encoding="utf-8"))

ESTOQUE_CD_NOVAMART = {
    "TENIS_ULTRABOOST_ROSA_37": {
        "sku": "TENIS_ULTRABOOST_ROSA_37",
        "produto": "Tênis de Corrida UltraBoost Rosa (Tam 37)",
        "unidades_disponiveis": 8,
        "status_estoque": "DISPONIVEL"
    },
    "BOTA_COURO_PRETA_39": {
        "sku": "BOTA_COURO_PRETA_39",
        "produto": "Bota de Couro Cano Curto Preta (Tam 39)",
        "unidades_disponiveis": 0,
        "status_estoque": "ESGOTADO"
    },
    "CAFETEIRA_BARISTA_PRO": {
        "sku": "CAFETEIRA_BARISTA_PRO",
        "produto": "Cafeteira Expresso Barista Pro 15 Bar",
        "unidades_disponiveis": 5,
        "status_estoque": "DISPONIVEL"
    }
}

PROTOCOLOS_REGISTRADOS_ERP = []

def _normalizar_texto(texto: str) -> str:
    sem_acento = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    return sem_acento.upper().strip()

@tool
def consultar_pedido_novamart(codigo_pedido: str) -> dict:
    """
    Consulta o banco de dados oficial da NovaMart para obter os dados reais de um pedido.
    Use esta ferramenta SEMPRE que o cliente mencionar um código de pedido (ex: '#NVM-4821', 'NVM-3091' ou '4821').
    Retorna nome do cliente, nível de fidelidade (Padrão ou VIP Prime), produto, categoria,
    status da entrega ('ENTREGUE' ou 'EM_TRANSITO') e o número exato de dias corridos desde a entrega.
    """
    codigo_limpo = codigo_pedido.strip().upper()
    if not codigo_limpo.startswith("#"):
        codigo_limpo = f"#{codigo_limpo}" if codigo_limpo.startswith("NVM-") else f"#NVM-{codigo_limpo}"

    pedido = BANCO_PEDIDOS_NOVAMART.get(codigo_limpo)
    if not pedido:
        return {
            "encontrado": False,
            "codigo_pesquisado": codigo_limpo,
            "mensagem_sistema": f"Pedido {codigo_limpo} não localizado na base de dados da NovaMart."
        }
    return {"encontrado": True, "dados_pedido": pedido}

@tool
def verificar_estoque_produto_cd(termo_busca_produto: str) -> dict:
    """
    Consulta o estoque do Centro de Distribuição (CD) da NovaMart para verificar se um produto,
    modelo ou tamanho específico está DISPONIVEL ou ESGOTADO para substituição/troca.
    Use SEMPRE antes de abrir um protocolo de troca de tamanho/cor para confirmar se o item desejado
    pelo cliente (ex: 'Bota Preta 39', 'Tênis Rosa 37', 'BOTA_COURO_PRETA_39') tem unidades disponíveis.
    """
    busca_norm = _normalizar_texto(termo_busca_produto)
    stopwords = {"DE", "DO", "DA", "NO", "NA", "PARA", "POR", "TAMANHO", "TAM", "NUMERO", "COR", "MODELO"}
    palavras_chave = [p for p in busca_norm.replace("_", " ").split() if p not in stopwords]

    if busca_norm in ESTOQUE_CD_NOVAMART:
        return {"encontrado": True, "item_estoque": ESTOQUE_CD_NOVAMART[busca_norm]}

    melhor_item = None
    maior_pontuacao = 0
    for sku, dados in ESTOQUE_CD_NOVAMART.items():
        alvo = _normalizar_texto(f"{sku} {dados['produto']}")
        pontos = sum(1 for p in palavras_chave if p in alvo)
        if pontos > maior_pontuacao:
            maior_pontuacao = pontos
            melhor_item = dados

    if melhor_item and maior_pontuacao > 0:
        return {
            "encontrado": True,
            "item_estoque": melhor_item,
            "orientacao_logistica": (
                "Produto disponível para reserva e troca imediata."
                if melhor_item["status_estoque"] == "DISPONIVEL"
                else "ATENÇÃO: Tamanho/modelo ESGOTADO no CD (0 unidades). Não abra protocolo de troca direta para este tamanho; ofereça Vale-Troca (liberado em até 24h após postagem) ou escolha de outro modelo/numeração."
            )
        }

    return {
        "encontrado": False,
        "termo_pesquisado": termo_busca_produto,
        "itens_cadastrados_no_cd": list(ESTOQUE_CD_NOVAMART.values())
    }

@tool
def abrir_protocolo_logistica_erp(
    codigo_pedido: str,
    tipo_operacao: Literal["POSTAGEM_GRATUITA_TROCA_DEVOLUCAO", "ORDEM_ANALISE_TECNICA_CD"],
    motivo_resumido: str
) -> dict:
    """
    Abre um protocolo oficial no sistema ERP de logística reversa da NovaMart e gera código de postagem dos Correios.
    IMPORTANTE:
    1. Só chame esta ferramenta DEPOIS de consultar o pedido em 'consultar_pedido_novamart' e confirmar que é ELEGÍVEL.
    2. Se for troca por novo tamanho/modelo, verifique ANTES em 'verificar_estoque_produto_cd' se o item desejado está DISPONIVEL.
       Se estiver ESGOTADO, só abra o protocolo ('POSTAGEM_GRATUITA_TROCA_DEVOLUCAO') DEPOIS que o cliente confirmar que aceita Vale-Troca ou outro item!
    3. Se for defeito funcional/técnico entre 8 e 90 dias corridos da entrega, use tipo_operacao='ORDEM_ANALISE_TECNICA_CD'.
    """
    codigo_limpo = codigo_pedido.strip().upper()
    if not codigo_limpo.startswith("#"):
        codigo_limpo = f"#{codigo_limpo}" if codigo_limpo.startswith("NVM-") else f"#NVM-{codigo_limpo}"

    pedido = BANCO_PEDIDOS_NOVAMART.get(codigo_limpo)
    if not pedido or pedido["status_entrega"] != "ENTREGUE":
        return {
            "sucesso": False,
            "erro": "Não é possível abrir protocolo de logística reversa para pedido inexistente ou ainda não entregue."
        }

    numero_sequencial = len(PROTOCOLOS_REGISTRADOS_ERP) + 501
    num_digits = codigo_limpo.replace("#NVM-", "")
    protocolo_id = f"PROT-2026-{numero_sequencial}"
    codigo_correios = f"BR-POST-{num_digits}-{numero_sequencial}"

    registro = {
        "protocolo": protocolo_id,
        "codigo_pedido": codigo_limpo,
        "cliente": pedido["cliente"],
        "nivel_fidelidade": pedido["nivel_fidelidade"],
        "tipo_operacao": tipo_operacao,
        "codigo_postagem_correios": codigo_correios,
        "motivo": motivo_resumido,
        "beneficio_vip_aplicado": pedido["nivel_fidelidade"] == "VIP Prime"
    }
    PROTOCOLOS_REGISTRADOS_ERP.append(registro)
    return {"sucesso": True, "detalhes_protocolo": registro}

ferramentas_dia5 = [
    consultar_pedido_novamart,
    verificar_estoque_produto_cd,
    abrir_protocolo_logistica_erp
]
mapa_ferramentas_dia5 = {f.name: f for f in ferramentas_dia5}

print(f"Ambiente operacional pronto: {len(ferramentas_dia5)} ferramentas carregadas!")

---
## Etapa 3: Repositório de Sessões (`ARMAZEM_SESSOES_CHAT`) e Janela de Contexto

Implementamos dois mecanismos para gerenciar diálogos de múltiplos turnos:

### 1. Isolamento de Sessão por `session_id` (`InMemoryChatMessageHistory`)
O dicionário `ARMAZEM_SESSOES_CHAT` associa cada `session_id` (ex: `"chat_fernanda_3091"`, `"chat_ricardo_8840"`) a uma instância independente de `InMemoryChatMessageHistory`.

### 2. Controle de Janela de Contexto (`recortar_janela_historico` / `trim_messages`)
Para evitar que conversas longas excedam o limite de tokens, recortamos o histórico garantindo duas regras:
1. **Preservação da `SystemMessage`** no início da lista de mensagens;
2. **Alinhamento em `HumanMessage` (`start_on="human"`):** Evita cortar o histórico no meio de um par `AIMessage(tool_calls)` $\rightarrow$ `ToolMessage`.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import (
    BaseMessage,
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
    trim_messages
)

# =====================================================================
# 1. Repositório Multi-Sessão (Isolamento por session_id)
# =====================================================================
ARMAZEM_SESSOES_CHAT: dict[str, InMemoryChatMessageHistory] = {}

def obter_historico_sessao(session_id: str) -> InMemoryChatMessageHistory:
    """
    Retorna o histórico de mensagens exclusivo de um 'session_id'.
    Se a sessão ainda não existir no armazém, cria uma nova instância limpa.
    """
    if session_id not in ARMAZEM_SESSOES_CHAT:
        ARMAZEM_SESSOES_CHAT[session_id] = InMemoryChatMessageHistory()
    return ARMAZEM_SESSOES_CHAT[session_id]


# =====================================================================
# 2. Estratégia de Janela Deslizante / Controle de Contexto
# =====================================================================
def recortar_janela_historico(
    mensagens_sessao: list[BaseMessage],
    max_mensagens_recentes: int = 16
) -> list[BaseMessage]:
    """
    Aplica uma janela deslizante sobre o histórico da sessão usando `trim_messages`:
    - Mantém no máximo as últimas `max_mensagens_recentes` mensagens;
    - Garante com `start_on='human'` que o recorte nunca comece em uma `ToolMessage`
      órfã ou no meio de uma chamada de ferramenta.
    """
    if len(mensagens_sessao) <= max_mensagens_recentes:
        return list(mensagens_sessao)

    return trim_messages(
        mensagens_sessao,
        max_tokens=max_mensagens_recentes,
        token_counter=len,  # Conta número de mensagens na janela
        strategy="last",
        start_on="human",
        include_system=True,
        allow_partial=False
    )

print("Repositório 'ARMAZEM_SESSOES_CHAT' e 'recortar_janela_historico' configurados com sucesso!")

---
## Etapa 4: Integrando `MessagesPlaceholder` e Memória por Sessão ao Motor ReAct Multi-Turno

Unimos o gerenciamento de histórico por `session_id` ao ciclo **ReAct**:

```text
┌───────────────────────────────────────────────────────────────────────────────────┐
│                FLUXO DO AGENTE REACT COM MEMÓRIA DE SESSÃO (DIA 5)                │
│                                                                                   │
│  [Nova Mensagem + session_id]                                                     │
│               │                                                                   │
│               ▼                                                                   │
│  ┌─────────────────────────┐                                                      │
│  │ obter_historico_sessao  │ -> Recupera turnos anteriores daquele session_id     │
│  │ + Janela (trim_messages)│ -> Mantém SystemMessage + últimas N interações       │
│  └────────────┬────────────┘                                                      │
│               │                                                                   │
│               ▼                                                                   │
│  ┌─────────────────────────┐                                                      │
│  │ Ciclo ReAct Multi-Turno │ -> PENSAMENTO -> AÇÃO (@tool) -> OBSERVAÇÃO          │
│  │ (Lembra dados prévios!) │    (Reaproveita informações dos turnos anteriores)   │
│  └────────────┬────────────┘                                                      │
│               │                                                                   │
│               ▼                                                                   │
│  ┌─────────────────────────┐                                                      │
│  │ Salva na Sessão         │ -> Grava HumanMessage, ToolCalls, ToolMessages e AI  │
│  └─────────────────────────┘                                                      │
└───────────────────────────────────────────────────────────────────────────────────┘
```

O `SYSTEM_PROMPT_MEMORIA_REACT_DIA5` inclui a diretriz **`<diretrizes_memoria_multi_turno>`**, permitindo que o agente solicite apenas o código `#NVM-XXXX` quando ausente e retome o objetivo do cliente assim que o código for informado no turno seguinte.

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

SYSTEM_PROMPT_MEMORIA_REACT_DIA5 = """
Você é a Clara, Agente Autônoma Sênior de Atendimento e Logística Reversa da NovaMart, operando no paradigma ReAct (Reasoning + Acting) com Memória Conversacional Multi-Turno.

<regras_de_negocio>
1. DIREITO DE ARREPENDIMENTO (Devolução com Reembolso Integral): Até 7 dias corridos após o recebimento (sem uso, na caixa original e com lacre/etiqueta intactos).
2. TROCA DE TAMANHO OU COR (Moda, Calçados e Acessórios):
   - Cliente 'Padrão': Prazo de até 30 dias corridos após o recebimento.
   - Cliente 'VIP Prime': Prazo estendido de até 60 dias corridos após o recebimento + liberação expressa no ato da postagem.
   - Condição obrigatória: Produto sem uso e com etiqueta afixada. A 1ª troca é gratuita.
   - REGRA DE ESTOQUE DO CD: Antes de abrir protocolo para troca de tamanho/modelo, você DEVE consultar 'verificar_estoque_produto_cd'.
     * Se o tamanho desejado estiver 'DISPONIVEL', abra o protocolo com tipo_operacao='POSTAGEM_GRATUITA_TROCA_DEVOLUCAO'.
     * Se o tamanho desejado estiver 'ESGOTADO' (0 unidades), NÃO abra o protocolo ainda! Informe que o pedido está dentro do prazo, avise que o tamanho desejado está esgotado no CD e pergunte se o cliente prefere: (a) devolver para receber um Vale-Troca NovaMart (liberado em até 24h após a postagem) ou (b) trocar por outro modelo/numeração.
     * Assim que o cliente confirmar no turno seguinte que deseja o Vale-Troca, abra o protocolo usando tipo_operacao='POSTAGEM_GRATUITA_TROCA_DEVOLUCAO' sem pedir o código do pedido novamente!
3. DEFEITO TÉCNICO OU DE FABRICAÇÃO:
   - Até 7 dias corridos do recebimento: Troca imediata.
   - De 8 a 90 dias corridos do recebimento: O produto deve ser encaminhado para análise técnica no Centro de Distribuição (prazo de laudo de até 15 dias úteis). Nesse caso, abra o protocolo usando obrigatoriamente tipo_operacao='ORDEM_ANALISE_TECNICA_CD'.
</regras_de_negocio>

<diretrizes_memoria_multi_turno>
- Você possui acesso ao histórico completo desta sessão ('historico').
- Se o cliente solicitar uma troca, devolução ou análise técnica, mas NÃO tiver informado o código do pedido (#NVM-XXXX) nem na mensagem atual nem nos turnos anteriores, NÃO invente um código: solicite educadamente o código do pedido (#NVM-XXXX) para prosseguir.
- Quando o cliente informar o código do pedido em um turno posterior, recupere do histórico qual era o produto/tamanho desejado, a condição informada (ex: sem uso e com etiqueta) ou o defeito relatado no turno anterior sem perguntar tudo de novo!
- Quando o pedido já tiver sido consultado em um turno anterior da mesma sessão, utilize o código e os dados já conhecidos para abrir o protocolo assim que o cliente autorizar.
</diretrizes_memoria_multi_turno>

<protocolo_react_obrigatorio>
- ANTES de acionar qualquer ferramenta (@tool), escreva SEMPRE no conteúdo de texto da sua mensagem um parágrafo curto iniciado exatamente por 'PENSAMENTO:' explicando o que você já sabe (incluindo o contexto dos turnos anteriores) e qual ferramenta vai acionar agora.
- Quando já tiver todas as observações necessárias (ou quando precisar apenas pedir o número do pedido ao cliente), NÃO escreva 'PENSAMENTO:': escreva diretamente a resposta final para o cliente.
</protocolo_react_obrigatorio>
"""

# Construindo o Prompt Template com MessagesPlaceholder para injetar o histórico da sessão
prompt_conversacional_dia5 = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_MEMORIA_REACT_DIA5),
    MessagesPlaceholder(variable_name="historico")
])

# Acoplando as 3 ferramentas ao LLM
llm_react_dia5 = llm.bind_tools(ferramentas_dia5)
cadeia_prompt_react_dia5 = prompt_conversacional_dia5 | llm_react_dia5

# Demonstrando também o empacotador declarativo RunnableWithMessageHistory do LangChain
agente_declarativo_com_historico = RunnableWithMessageHistory(
    prompt_conversacional_dia5 | llm_react_dia5,
    obter_historico_sessao,
    input_messages_key="historico"
)

def _extrair_texto_limpo(conteudo) -> str:
    """Extrai o texto limpo de AIMessage.content (seja string ou lista de blocos do Gemini)."""
    if isinstance(conteudo, str):
        return conteudo.strip()
    if isinstance(conteudo, list):
        partes = []
        for bloco in conteudo:
            if isinstance(bloco, dict) and bloco.get("type") == "text":
                partes.append(bloco.get("text", ""))
            elif isinstance(bloco, str):
                partes.append(bloco)
        return "\n".join(partes).strip()
    return str(conteudo).strip()


def conversar_com_clara_dia5(
    mensagem_cliente: str,
    session_id: str,
    max_iter: int = 6,
    max_mensagens_janela: int = 16,
    verbose: bool = True
) -> dict:
    """
    Motor ReAct Conversacional Multi-Sessão (Dia 5):
    1. Recupera o histórico isolado do `session_id` via `obter_historico_sessao`;
    2. Adiciona a nova `HumanMessage` do cliente à sessão;
    3. Aplica o controle de janela (`recortar_janela_historico`) mantendo o contexto enxuto;
    4. Executa o ciclo ReAct (Pensamento -> Ação -> Observação) registrando todas as mensagens
       e retornos de ferramentas no histórico da sessão para que os próximos turnos lembrem de tudo!
    """
    historico_sessao = obter_historico_sessao(session_id)
    historico_sessao.add_message(HumanMessage(content=mensagem_cliente))

    trilha_auditoria_turno = []

    if verbose:
        print(f"💬 [SESSÃO: {session_id}] 🗣️ [CLIENTE]: {mensagem_cliente}")
        print("-" * 95)

    for iteracao in range(1, max_iter + 1):
        # Aplica a janela deslizante de contexto sobre as mensagens da sessão
        historico_recortado = recortar_janela_historico(
            historico_sessao.messages,
            max_mensagens_recentes=max_mensagens_janela
        )

        # Invoca o modelo com SystemPrompt + MessagesPlaceholder(historico)
        resposta_ia = cadeia_prompt_react_dia5.invoke({"historico": historico_recortado})
        historico_sessao.add_message(resposta_ia)

        texto_ia = _extrair_texto_limpo(resposta_ia.content)

        # Se não há chamadas de ferramentas, a Clara chegou à resposta final deste turno!
        if not resposta_ia.tool_calls:
            if verbose:
                print(f"🤖 [CLARA ({session_id} — Iteração {iteracao})]:")
                print(texto_ia)
            return {
                "session_id": session_id,
                "iteracoes_utilizadas": iteracao,
                "total_mensagens_na_sessao": len(historico_sessao.messages),
                "mensagens_enviadas_na_janela": len(historico_recortado),
                "trilha_react_turno": trilha_auditoria_turno,
                "resposta_final": texto_ia
            }

        # Caso haja tool_calls, registramos o Pensamento -> Ação -> Observação
        pensamento_passo = texto_ia if texto_ia else (
            f"PENSAMENTO: Acionando {[c['name'] for c in resposta_ia.tool_calls]} com base no contexto da sessão '{session_id}'."
        )
        if verbose:
            print(f"🧠 [ITERAÇÃO {iteracao} — PENSAMENTO (Thought)]:")
            print(f"   {pensamento_passo}")

        for chamada in resposta_ia.tool_calls:
            nome_tool = chamada["name"]
            args_tool = chamada["args"]

            if verbose:
                print(f"🛠️ [ITERAÇÃO {iteracao} — AÇÃO (Action)]:")
                print(f"   -> Tool: {nome_tool}({json.dumps(args_tool, ensure_ascii=False)})")

            resultado_tool = mapa_ferramentas_dia5[nome_tool].invoke(args_tool)

            if verbose:
                print(f"👁️ [ITERAÇÃO {iteracao} — OBSERVAÇÃO (Observation)]:")
                print(f"   <- Retorno: {json.dumps(resultado_tool, ensure_ascii=False)}")

            trilha_auditoria_turno.append({
                "iteracao": iteracao,
                "pensamento": pensamento_passo,
                "acao_ferramenta": nome_tool,
                "argumentos": args_tool,
                "observacao_retorno": resultado_tool
            })

            # Crucial: salvamos a ToolMessage na sessão para que os próximos turnos lembrem dos dados consultados!
            historico_sessao.add_message(
                ToolMessage(
                    content=json.dumps(resultado_tool, ensure_ascii=False),
                    tool_call_id=chamada["id"],
                    name=nome_tool
                )
            )

    # Fechamento de segurança caso atinja max_iter
    mensagens_finais = prompt_conversacional_dia5.format_messages(
        historico=recortar_janela_historico(historico_sessao.messages, max_mensagens_janela)
    )
    resposta_fechamento = llm.invoke(mensagens_finais)
    historico_sessao.add_message(resposta_fechamento)
    texto_final = _extrair_texto_limpo(resposta_fechamento.content)
    return {
        "session_id": session_id,
        "iteracoes_utilizadas": max_iter,
        "total_mensagens_na_sessao": len(historico_sessao.messages),
        "mensagens_enviadas_na_janela": len(mensagens_finais),
        "trilha_react_turno": trilha_auditoria_turno,
        "resposta_final": texto_final
    }

print("Motor 'conversar_com_clara_dia5' configurado com memória multi-sessão!")

---
## Etapa 5: Simulação Multi-Turno e Teste de Isolamento de Sessão

Validamos o agente conversacional em duas sessões independentes:

### Simulação 1: `session_id = "chat_fernanda_3091"` (3 Turnos Conectados)
- **Turno 1:** Cliente solicita troca de bota preta sem uso pelo **tamanho 39**, sem informar o número do pedido.
- **Turno 2:** Cliente informa `"O número do meu pedido é #NVM-3091."` O agente recupera da memória que a troca desejada é para o tamanho 39, consulta `#NVM-3091`, verifica o estoque (`ESGOTADO`) e oferece Vale-Troca.
- **Turno 3:** Cliente aceita o Vale-Troca e pede o código dos Correios. O agente recupera `#NVM-3091` da memória da sessão e aciona `abrir_protocolo_logistica_erp`.

### Simulação 2: `session_id = "chat_ricardo_8840"` (Sessão Paralela Isolada)
- **Turno 1:** Cliente relata defeito na cafeteira expresso após 25 dias (sem código do pedido).
- **Turno 2:** Cliente informa `#NVM-8840` e solicita a abertura da análise técnica (`ORDEM_ANALISE_TECNICA_CD`).

In [ ]:
# Limpando o armazém de sessões e a lista de protocolos antes de iniciar a simulação oficial
ARMAZEM_SESSOES_CHAT.clear()
PROTOCOLOS_REGISTRADOS_ERP.clear()

roteiro_fernanda = [
    "Oi Clara! Comprei uma bota de couro preta com vocês, não usei e está com a etiqueta na caixa, mas ficou apertada. Quero trocar pelo tamanho 39!",
    "O número do meu pedido é #NVM-3091.",
    "Puxa, que pena que a 39 esgotou! Nesse caso, quero devolver para receber um Vale-Troca NovaMart. Pode abrir o protocolo e gerar meu código dos Correios?"
]

print("=" * 95)
print("🟢 INICIANDO SIMULAÇÃO 1 — SESSÃO: 'chat_fernanda_3091' (3 TURNOS CONECTADOS)")
print("=" * 95)

resultados_fernanda = []
for turno_idx, msg in enumerate(roteiro_fernanda, 1):
    print(f"\n--- [TURNO {turno_idx} | SESSÃO: chat_fernanda_3091] ---")
    res = conversar_com_clara_dia5(
        mensagem_cliente=msg,
        session_id="chat_fernanda_3091",
        verbose=True
    )
    resultados_fernanda.append(res)

In [ ]:
roteiro_ricardo = [
    "Olá, meu nome é Ricardo e minha cafeteira expresso parou de esquentar água após 25 dias.",
    "O pedido dela é o #NVM-8840. Pode abrir a ordem de análise técnica para mim?"
]

print("=" * 95)
print("🔵 INICIANDO SIMULAÇÃO 2 — SESSÃO: 'chat_ricardo_8840' (ISOLAMENTO DE SESSÃO)")
print("=" * 95)

resultados_ricardo = []
for turno_idx, msg in enumerate(roteiro_ricardo, 1):
    print(f"\n--- [TURNO {turno_idx} | SESSÃO: chat_ricardo_8840] ---")
    res = conversar_com_clara_dia5(
        mensagem_cliente=msg,
        session_id="chat_ricardo_8840",
        verbose=True
    )
    resultados_ricardo.append(res)

### Auditoria das Sessões de Memória (`ARMAZEM_SESSOES_CHAT`) e Protocolos no ERP

Vamos inspecionar o estado interno do `ARMAZEM_SESSOES_CHAT` e do `PROTOCOLOS_REGISTRADOS_ERP` para comprovar:
1. Que as duas sessões (`chat_fernanda_3091` e `chat_ricardo_8840`) estão **100% separadas** na memória;
2. Que todos os objetos (`HumanMessage`, `AIMessage` e `ToolMessage`) foram preservados na ordem correta;
3. Que a função `recortar_janela_historico` mantém a integridade dos turnos quando um limite menor de janela é testado;
4. Que os **2 protocolos oficiais** (`#NVM-3091` para Vale-Troca da Fernanda no Turno 3 e `#NVM-8840` para Análise Técnica do Ricardo no Turno 2) foram registrados no ERP!

In [ ]:
print("=== 1. AUDITORIA DE ISOLAMENTO NO ARMAZEM_SESSOES_CHAT ===")
for sid, hist in ARMAZEM_SESSOES_CHAT.items():
    tipos_msgs = [type(m).__name__ for m in hist.messages]
    print(f" -> Sessão '{sid}': {len(hist.messages)} mensagens armazenadas | Sequência: {tipos_msgs}")

print("\n=== 2. TESTE DA PROTEÇÃO DE JANELA DE CONTEXTO (recortar_janela_historico) ===")
historico_fernanda_completo = obter_historico_sessao("chat_fernanda_3091").messages
janela_curta_teste = recortar_janela_historico(historico_fernanda_completo, max_mensagens_recentes=4)
print(f" -> Histórico total da Fernanda: {len(historico_fernanda_completo)} mensagens")
print(f" -> Após recorte seguro (max_mensagens_recentes=4, start_on='human'): {len(janela_curta_teste)} mensagens")
print(f" -> Tipo da 1ª mensagem da janela recortada: {type(janela_curta_teste[0]).__name__} (garante que não começa em ToolMessage órfã!)")

print(f"\n=== 3. PROTOCOLOS GERADOS NO ERP ({len(PROTOCOLOS_REGISTRADOS_ERP)} protocolos) ===")
print(json.dumps(PROTOCOLOS_REGISTRADOS_ERP, indent=2, ensure_ascii=False))

---
## Resumo da Implementação — Dia 5 Concluído

### Componentes construídos neste notebook:
- Repositório multi-sessão **`ARMAZEM_SESSOES_CHAT`** e função **`obter_historico_sessao(session_id)`** com `InMemoryChatMessageHistory`;
- Controle de janela deslizante **`recortar_janela_historico`** com `trim_messages(..., start_on="human")`;
- Integração de histórico conversacional ao loop **ReAct (`conversar_com_clara_dia5`)**, mantendo o contexto entre turnos e o isolamento entre sessões paralelas.

Compartilhe seu progresso nas redes sociais com a hashtag **#7DaysOfCode**!